In [1]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")

tables = {}

for file in sorted(RAW_DIR.glob("*.csv")):
    df = pd.read_csv(file)
    tables[file.stem] = df
    
    print(f"{file.name}")
    print(f"  Rows    : {len(df):,}")
    print(f"  Columns : {len(df.columns)}")
    print(f"  Missing : {df.isna().sum().sum():,}")
    print(f"  Duplicate rows: {df.duplicated().sum():,}")
    print("-" * 60)

admission.csv
  Rows    : 45,000
  Columns : 10
  Missing : 0
  Duplicate rows: 0
------------------------------------------------------------
bed.csv
  Rows    : 415
  Columns : 4
  Missing : 0
  Duplicate rows: 0
------------------------------------------------------------
billing.csv
  Rows    : 45,000
  Columns : 8
  Missing : 0
  Duplicate rows: 0
------------------------------------------------------------
billing_detail.csv
  Rows    : 112,402
  Columns : 5
  Missing : 67,402
  Duplicate rows: 0
------------------------------------------------------------
department.csv
  Rows    : 11
  Columns : 5
  Missing : 0
  Duplicate rows: 0
------------------------------------------------------------
diagnostic_test.csv
  Rows    : 9
  Columns : 5
  Missing : 0
  Duplicate rows: 0
------------------------------------------------------------
disease.csv
  Rows    : 20
  Columns : 3
  Missing : 0
  Duplicate rows: 0
------------------------------------------------------------
doctor.csv
  

In [2]:
billing_detail = tables["billing_detail"]

print("Missing values by column:")
print(billing_detail.isna().sum())

print("\nSample rows with missing values:")
print(billing_detail[billing_detail.isna().any(axis=1)].head(10))

Missing values by column:
billing_detail_id        0
charge_type              0
reference_id         67402
amount                   0
bill_id                  0
dtype: int64

Sample rows with missing values:
    billing_detail_id charge_type  reference_id  amount  bill_id
1                   2        Drug           NaN    1844        1
3                   4        Test           NaN     562        2
4                   5        Drug           NaN    2489        2
5                   6   Procedure           NaN    6205        2
7                   8        Test           NaN    1726        3
8                   9        Drug           NaN     615        3
12                 13   Procedure           NaN   17891        6
15                 16        Test           NaN     653        8
16                 17        Drug           NaN     370        8
18                 19        Drug           NaN     585        9


In [3]:
print(billing_detail["charge_type"].value_counts(dropna=False))


charge_type
Room         45000
Drug         31344
Test         22483
Procedure    13575
Name: count, dtype: int64


In [4]:
print(
    billing_detail.groupby("charge_type")["reference_id"]
    .apply(lambda x: x.notna().sum())
)

charge_type
Drug             0
Procedure        0
Room         45000
Test             0
Name: reference_id, dtype: int64


In [5]:
print(billing_detail[billing_detail["charge_type"] == "Room"].head(10))


    billing_detail_id charge_type  reference_id  amount  bill_id
0                   1        Room          76.0    4884        1
2                   3        Room         302.0   70010        2
6                   7        Room          11.0   40230        3
9                  10        Room         128.0   10445        4
10                 11        Room         157.0   23355        5
11                 12        Room         143.0   50120        6
13                 14        Room          79.0    7225        7
14                 15        Room         276.0   20346        8
17                 18        Room          55.0   17070        9
20                 21        Room          11.0   21896       10


In [6]:
room_refs = billing_detail.loc[
    billing_detail["charge_type"] == "Room",
    "reference_id"
].dropna().astype(int)

bed_ids = set(tables["bed"]["bed_id"])

print("Room reference IDs:", len(room_refs))
print("Unique Room reference IDs:", room_refs.nunique())
print("Matching bed IDs:", room_refs.isin(bed_ids).sum())
print("Non-matching IDs:", (~room_refs.isin(bed_ids)).sum())

Room reference IDs: 45000
Unique Room reference IDs: 145
Matching bed IDs: 45000
Non-matching IDs: 0


In [7]:
# Referential integrity checks

checks = {
    "admission → patient": (
        tables["admission"]["patient_id"].isin(tables["patient"]["patient_id"]).mean()
    ),
    "admission → department": (
        tables["admission"]["department_id"].isin(tables["department"]["department_id"]).mean()
    ),
    "admission → ward": (
        tables["admission"]["ward_id"].isin(tables["ward"]["ward_id"]).mean()
    ),
    "admission → bed": (
        tables["admission"]["bed_id"].isin(tables["bed"]["bed_id"]).mean()
    ),
    "admission → disease": (
        tables["admission"]["disease_id"].isin(tables["disease"]["disease_id"]).mean()
    ),
    "billing → admission": (
        tables["billing"]["admission_id"].isin(tables["admission"]["admission_id"]).mean()
    )
}

for relationship, match_rate in checks.items():
    print(f"{relationship}: {match_rate:.2%}")

admission → patient: 100.00%
admission → department: 100.00%
admission → ward: 100.00%
admission → bed: 100.00%
admission → disease: 100.00%
billing → admission: 100.00%


In [8]:
# Check admission date consistency

admission = tables["admission"].copy()

admission["admission_date"] = pd.to_datetime(
    admission["admission_date"], errors="coerce"
)

admission["discharge_date"] = pd.to_datetime(
    admission["discharge_date"], errors="coerce"
)

invalid_dates = admission[
    admission["discharge_date"] < admission["admission_date"]
]

print("Invalid date records:", len(invalid_dates))

print("\nAdmission date range:")
print(admission["admission_date"].min(), "to", admission["admission_date"].max())

print("\nDischarge date range:")
print(admission["discharge_date"].min(), "to", admission["discharge_date"].max())

Invalid date records: 0

Admission date range:
2020-01-01 00:00:00 to 2025-12-31 00:00:00

Discharge date range:
2020-01-02 00:00:00 to 2026-01-12 00:00:00


In [9]:
print("Admission Types:")
print(tables["admission"]["admission_type"].value_counts(dropna=False))

print("\nAdmission Status:")
print(tables["admission"]["admission_status"].value_counts(dropna=False))


Admission Types:
admission_type
Elective     26923
Emergency    18077
Name: count, dtype: int64

Admission Status:
admission_status
Discharged    45000
Name: count, dtype: int64


In [10]:
print("Bed Status:")
print(tables["bed"]["bed_status"].value_counts(dropna=False))

print("\nWard Types:")
print(tables["ward"]["ward_type"].value_counts(dropna=False))

Bed Status:
bed_status
Occupied     270
Available    145
Name: count, dtype: int64

Ward Types:
ward_type
General         14
Private          5
Semi-Private     4
ICU              4
Name: count, dtype: int64


In [11]:
print("Department Status:")
print(tables["department"]["status"].value_counts(dropna=False))

print("\nDepartment Types:")
print(tables["department"]["department_type"].value_counts(dropna=False))

Department Status:
status
Active    11
Name: count, dtype: int64

Department Types:
department_type
Clinical      6
Admin         3
Diagnostic    2
Name: count, dtype: int64


In [12]:
print("Billing amount ranges:")
print(tables["billing"][[
    "total_amount",
    "insurance_covered_amount",
    "patient_payable_amount"
]].describe())

print("\nPatient insurance coverage:")
print(tables["patient_insurance"]["coverage_percentage"].describe())

print("\nWard bed capacity:")
print(tables["ward"]["total_beds"].describe())

print("\nDoctor experience:")
print(tables["doctor"]["experience_years"].describe())

Billing amount ranges:
       total_amount  insurance_covered_amount  patient_payable_amount
count  45000.000000              45000.000000            45000.000000
mean   37427.691311              21708.463711            15719.227600
std    20898.321091              18289.872062            16216.113367
min     5001.000000                  0.000000              500.300000
25%    21231.000000               6580.000000             4743.850000
50%    34623.500000              19210.500000             9712.150000
75%    48039.250000              32752.325000            20746.750000
max    89990.000000              80956.800000            89921.000000

Patient insurance coverage:
count    21617.000000
mean        70.069852
std         14.160107
min         50.000000
25%         60.000000
50%         70.000000
75%         80.000000
max         90.000000
Name: coverage_percentage, dtype: float64

Ward bed capacity:
count    27.000000
mean     15.370370
std       4.143096
min      10.000000
25% 

In [13]:
billing = tables["billing"].copy()

billing["calculated_total"] = (
    billing["insurance_covered_amount"]
    + billing["patient_payable_amount"]
)

difference = (
    billing["total_amount"] - billing["calculated_total"]
).abs()

print("Bills:", len(billing))
print("Exact matches:", (difference == 0).sum())
print("Non-matching bills:", (difference != 0).sum())
print("Maximum difference:", difference.max())
print("Average difference:", difference.mean())

Bills: 45000
Exact matches: 45000
Non-matching bills: 0
Maximum difference: 0.0
Average difference: 0.0


In [14]:
admission = tables["admission"].copy()

admission["admission_date"] = pd.to_datetime(admission["admission_date"])
admission["discharge_date"] = pd.to_datetime(admission["discharge_date"])

admission["length_of_stay_days"] = (
    admission["discharge_date"] - admission["admission_date"]
).dt.days

print(admission["length_of_stay_days"].describe())
print("\nZero-day stays:", (admission["length_of_stay_days"] == 0).sum())
print("Negative stays:", (admission["length_of_stay_days"] < 0).sum())

count    45000.000000
mean         5.155000
std          3.271284
min          1.000000
25%          3.000000
50%          4.000000
75%          8.000000
max         15.000000
Name: length_of_stay_days, dtype: float64

Zero-day stays: 0
Negative stays: 0


In [15]:
# Create a clean working copy of the admission table

admission_clean = tables["admission"].copy()

# Convert dates
admission_clean["admission_date"] = pd.to_datetime(
    admission_clean["admission_date"], errors="coerce"
)

admission_clean["discharge_date"] = pd.to_datetime(
    admission_clean["discharge_date"], errors="coerce"
)

# Create Length of Stay
admission_clean["length_of_stay_days"] = (
    admission_clean["discharge_date"]
    - admission_clean["admission_date"]
).dt.days

# Check the result
print(admission_clean.head())
print("\nMissing values:")
print(admission_clean.isna().sum())

   admission_id admission_date discharge_date admission_type admission_status  \
0             1     2020-02-25     2020-02-27      Emergency       Discharged   
1             2     2022-02-22     2022-03-04       Elective       Discharged   
2             3     2021-02-03     2021-02-09       Elective       Discharged   
3             4     2021-12-31     2022-01-05       Elective       Discharged   
4             5     2022-07-02     2022-07-07       Elective       Discharged   

   patient_id  department_id  ward_id  bed_id  disease_id  length_of_stay_days  
0         166              2        6      76          10                    2  
1        8622              5       21     302          11                   10  
2       23976              1        2      11           9                    6  
3       16635              2       10     128           1                    5  
4       10654              3       11     157           7                    5  

Missing values:
admission_

In [16]:
# Create a clean working copy of the patient table

patient_clean = tables["patient"].copy()

# Convert date of birth
patient_clean["date_of_birth"] = pd.to_datetime(
    patient_clean["date_of_birth"], errors="coerce"
)

# Check the result
print(patient_clean.head())

print("\nMissing values:")
print(patient_clean.isna().sum())

print("\nDuplicate patient IDs:")
print(patient_clean["patient_id"].duplicated().sum())

   patient_id  gender date_of_birth blood_group                city  \
0           1  Female    1987-08-24          O-  East Stephanieberg   
1           2    Male    1960-05-18          A-          Manuelbury   
2           3    Male    1955-04-24          A-   Lake Susanchester   
3           4    Male    2004-06-16          B-   South Leslieburgh   
4           5    Male    1977-07-22          A-        Lopezchester   

        contact_number  
0      +1-792-342-0981  
1         793-725-0800  
2  +1-330-617-3749x232  
3   426-611-6235x07684  
4   (215)330-2831x0821  

Missing values:
patient_id        0
gender            0
date_of_birth     0
blood_group       0
city              0
contact_number    0
dtype: int64

Duplicate patient IDs:
0


In [17]:
# Clean department, ward and bed tables

department_clean = tables["department"].copy()
ward_clean = tables["ward"].copy()
bed_clean = tables["bed"].copy()

print("Department missing values:")
print(department_clean.isna().sum())

print("\nWard missing values:")
print(ward_clean.isna().sum())

print("\nBed missing values:")
print(bed_clean.isna().sum())

print("\nDuplicate Department IDs:",
      department_clean["department_id"].duplicated().sum())

print("Duplicate Ward IDs:",
      ward_clean["ward_id"].duplicated().sum())

print("Duplicate Bed IDs:",
      bed_clean["bed_id"].duplicated().sum())

Department missing values:
department_id      0
department_name    0
department_type    0
floor_number       0
status             0
dtype: int64

Ward missing values:
ward_id          0
ward_name        0
ward_type        0
total_beds       0
department_id    0
dtype: int64

Bed missing values:
bed_id        0
bed_number    0
bed_status    0
ward_id       0
dtype: int64

Duplicate Department IDs: 0
Duplicate Ward IDs: 0
Duplicate Bed IDs: 0


In [18]:
# Clean disease, doctor and employee tables

disease_clean = tables["disease"].copy()
doctor_clean = tables["doctor"].copy()
employee_clean = tables["employee"].copy()

print("Disease missing values:")
print(disease_clean.isna().sum())

print("\nDoctor missing values:")
print(doctor_clean.isna().sum())

print("\nEmployee missing values:")
print(employee_clean.isna().sum())

print("\nDuplicate Disease IDs:",
      disease_clean["disease_id"].duplicated().sum())

print("Duplicate Doctor IDs:",
      doctor_clean["doctor_id"].duplicated().sum())

print("Duplicate Employee IDs:",
      employee_clean["employee_id"].duplicated().sum())

Disease missing values:
disease_id          0
disease_name        0
disease_category    0
dtype: int64

Doctor missing values:
doctor_id           0
employee_id         0
specialization      0
qualification       0
experience_years    0
dtype: int64

Employee missing values:
employee_id        0
employee_name      0
gender             0
role               0
employment_type    0
date_of_joining    0
department_id      0
dtype: int64

Duplicate Disease IDs: 0
Duplicate Doctor IDs: 0
Duplicate Employee IDs: 0


In [19]:
# Clean billing and billing detail tables

billing_clean = tables["billing"].copy()
billing_detail_clean = tables["billing_detail"].copy()

print("Billing missing values:")
print(billing_clean.isna().sum())

print("\nBilling detail missing values:")
print(billing_detail_clean.isna().sum())

print("\nDuplicate Bill IDs:",
      billing_clean["bill_id"].duplicated().sum())

print("Duplicate Billing Detail IDs:",
      billing_detail_clean["billing_detail_id"].duplicated().sum())

Billing missing values:
bill_id                     0
bill_date                   0
total_amount                0
insurance_covered_amount    0
patient_payable_amount      0
payment_status              0
payment_mode                0
admission_id                0
dtype: int64

Billing detail missing values:
billing_detail_id        0
charge_type              0
reference_id         67402
amount                   0
bill_id                  0
dtype: int64

Duplicate Bill IDs: 0
Duplicate Billing Detail IDs: 0


In [20]:
# Clean diagnostic tables

diagnostic_test_clean = tables["diagnostic_test"].copy()
patient_diagnostic_clean = tables["patient_diagnostic"].copy()

print("Diagnostic test missing values:")
print(diagnostic_test_clean.isna().sum())

print("\nPatient diagnostic missing values:")
print(patient_diagnostic_clean.isna().sum())

print("\nDuplicate Test IDs:",
      diagnostic_test_clean["test_id"].duplicated().sum())

print("Duplicate Patient Diagnostic IDs:",
      patient_diagnostic_clean["patient_diagnostic_id"].duplicated().sum())

Diagnostic test missing values:
test_id          0
test_name        0
test_category    0
standard_cost    0
department_id    0
dtype: int64

Patient diagnostic missing values:
patient_diagnostic_id    0
test_date                0
result_status            0
admission_id             0
test_id                  0
doctor_id                0
dtype: int64

Duplicate Test IDs: 0
Duplicate Patient Diagnostic IDs: 0


In [21]:
# Clean prescription and drug tables

prescription_clean = tables["prescription"].copy()
drug_clean = tables["drug"].copy()

print("Prescription missing values:")
print(prescription_clean.isna().sum())

print("\nDrug missing values:")
print(drug_clean.isna().sum())

print("\nDuplicate Prescription IDs:",
      prescription_clean["prescription_id"].duplicated().sum())

print("Duplicate Drug IDs:",
      drug_clean["drug_id"].duplicated().sum())

Prescription missing values:
prescription_id    0
dosage             0
frequency          0
duration_days      0
admission_id       0
drug_id            0
dtype: int64

Drug missing values:
drug_id            0
drug_name          0
brand_name         0
drug_category      0
unit_cost          0
manufacturer_id    0
dtype: int64

Duplicate Prescription IDs: 0
Duplicate Drug IDs: 0


In [22]:
# Clean drug inventory and manufacturer tables

drug_inventory_clean = tables["drug_inventory"].copy()
drug_manufacturer_clean = tables["drug_manufacturer"].copy()

print("Drug inventory missing values:")
print(drug_inventory_clean.isna().sum())

print("\nDrug manufacturer missing values:")
print(drug_manufacturer_clean.isna().sum())

print("\nDuplicate Drug Inventory IDs:",
      drug_inventory_clean["drug_id"].duplicated().sum())

print("Duplicate Manufacturer IDs:",
      drug_manufacturer_clean["manufacturer_id"].duplicated().sum())

Drug inventory missing values:
inventory_id         0
current_stock        0
reorder_level        0
inventory_status     0
last_restock_date    0
drug_id              0
dtype: int64

Drug manufacturer missing values:
manufacturer_id       0
manufacturer_name     0
country               0
reliability_rating    0
contract_status       0
dtype: int64

Duplicate Drug Inventory IDs: 0
Duplicate Manufacturer IDs: 0


In [23]:
# Clean insurance and staff assignment tables

patient_insurance_clean = tables["patient_insurance"].copy()
insurance_provider_clean = tables["insurance_provider"].copy()
staff_assignment_clean = tables["staff_assignment"].copy()

print("Patient insurance missing values:")
print(patient_insurance_clean.isna().sum())

print("\nInsurance provider missing values:")
print(insurance_provider_clean.isna().sum())

print("\nStaff assignment missing values:")
print(staff_assignment_clean.isna().sum())

print("\nDuplicate Patient Insurance IDs:",
      patient_insurance_clean["patient_insurance_id"].duplicated().sum())

print("Duplicate Insurance Provider IDs:",
      insurance_provider_clean["insurance_provider_id"].duplicated().sum())

print("Duplicate Assignment IDs:",
      staff_assignment_clean["assignment_id"].duplicated().sum())

Patient insurance missing values:
patient_insurance_id     0
policy_number            0
coverage_percentage      0
policy_start_date        0
policy_end_date          0
patient_id               0
insurance_provider_id    0
dtype: int64

Insurance provider missing values:
insurance_provider_id    0
provider_name            0
provider_type            0
contact_details          0
coverage_limit           0
dtype: int64

Staff assignment missing values:
assignment_id    0
employee_id      0
ward_id          0
shift            0
dtype: int64

Duplicate Patient Insurance IDs: 0
Duplicate Insurance Provider IDs: 0
Duplicate Assignment IDs: 0


In [24]:
admission_ids = set(tables["admission"]["admission_id"])

billing_ids = set(tables["billing"]["admission_id"])
diagnostic_ids = set(tables["patient_diagnostic"]["admission_id"])
prescription_ids = set(tables["prescription"]["admission_id"])

# Admissions with patient insurance
insured_patient_ids = set(tables["patient_insurance"]["patient_id"])
admission_patient_ids = tables["admission"]["patient_id"]

print("Total admissions:", len(admission_ids))

print(
    "Admissions with billing:",
    tables["billing"]["admission_id"].isin(admission_ids).sum()
)

print(
    "Admissions with diagnostics:",
    tables["patient_diagnostic"]["admission_id"].isin(admission_ids).sum()
)

print(
    "Admissions with prescriptions:",
    tables["prescription"]["admission_id"].isin(admission_ids).sum()
)

print(
    "Admissions whose patients have insurance:",
    admission_patient_ids.isin(insured_patient_ids).sum()
)

Total admissions: 45000
Admissions with billing: 45000
Admissions with diagnostics: 63269
Admissions with prescriptions: 73109
Admissions whose patients have insurance: 27111
